# Audit a spglib reciprocal mesh

Install `hecke-mass[crystal]` in this kernel's environment. From a repository checkout, use `python -m pip install './python[crystal]'`.

This recipe checks the mesh partition against reciprocal rotations selected by spglib for the supplied cell.

In [ ]:
from fractions import Fraction

from hecke_mass.adapters import audit_spglib_mesh

cell = ([[1, 0, 0], [0, 1, 0], [0, 0, 1]], [[0, 0, 0]], [1])
report = audit_spglib_mesh(cell, (2, 2, 2))
assert report.audit.status == "complete"
assert report.group_order == 48
assert sorted(orbit.multiplicity for orbit in report.orbits) == [1, 1, 3, 3]
[(orbit.address, orbit.stabilizer_order, orbit.weight) for orbit in report.orbits]

In [ ]:
# The observable is one on the orbit with exactly one odd coordinate.
values = [int(sum(x % 2 for x in orbit.address) == 1) for orbit in report.orbits]
average = report.weighted_average(values)
assert average == Fraction(3, 8)
average

## A shifted anisotropic mesh

Rotations must preserve the mesh dimensions and shift. The adapter computes the group for this mesh from the supplied crystal, including operations that act trivially on individual grid points.

In [ ]:
shifted = audit_spglib_mesh(cell, (4, 2, 2), shift=(1, 0, 0))
assert shifted.audit.status == "complete"
assert shifted.audit.covered == 16
assert sum(orbit.weight for orbit in shifted.orbits) == 1
shifted.group_order

The exact checks concern the finite action determined by the supplied rotations. The cell and `symprec` still govern which symmetries spglib finds.